# Playbook Soccer Analytics — Colab (GPU)

**One-time setup before running:** `Runtime → Change runtime type → T4 GPU`.

Then run the cells **top to bottom, once each**. No repair cells, no restarts, no CPU/GPU switching.

What you need:
- A **Roboflow API key** saved in Colab Secrets (🔑 left sidebar) as `ROBOFLOW_API_KEY`.
- A soccer clip (`.mp4`).

The repo branch `claude/setup-gpu-video-testing-JhgUH` already carries the correct config
(stateless homography, fine-tuned field model `f07vi-it2xv/3`, `KP_CONF=0.50`).
**This notebook does NOT patch any repo files** — that was the source of past breakage.

In [ ]:
# Cell 1 — Verify GPU
import subprocess
g = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if g.returncode == 0:
    print('GPU:', g.stdout.strip())
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.')

In [ ]:
# Cell 2 — Install everything (GPU build), once.
# Order matters: install the CUDA onnxruntime LAST so nothing shadows it with the
# CPU build. This is what removes the old repair/restart dance.
import sys

# System libs for OpenCV/ffmpeg
!apt-get install -qq ffmpeg libglib2.0-0 libsm6 libxext6 libxrender-dev >/dev/null

# Colab ships opencv-python; swap to headless to avoid display-backend conflicts
!pip uninstall -qqy opencv-python opencv-python-headless >/dev/null 2>&1

!pip install -q \
    'numpy>=2.0.0,<2.4.0' \
    opencv-python-headless==4.10.0.84 \
    tqdm 'requests>=2.32.3' \
    'pydantic>=2.11.7,<2.12.0' pydantic-settings==2.4.0 python-dotenv==1.0.1 \
    'supervision==0.27.0.post2' \
    'ultralytics>=8.4.37,<8.5.0' 'lap>=0.5.13,<0.6' \
    'transformers>=5.2.0,<5.3.0'

# Roboflow sports (pitch config, annotators, color-team helper)
!pip install -q git+https://github.com/roboflow/sports.git@main

# Roboflow inference — GPU variant
!pip install -q inference-gpu==1.3.0

# CUDA 12 onnxruntime LAST (wins over any CPU onnxruntime pulled in as a dep)
!pip install -q onnxruntime-gpu==1.20.1 \
    --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/

import onnxruntime as ort
print('\nonnxruntime providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDA EP missing — re-run this cell.'
print('OK — GPU inference ready.')

In [ ]:
# Cell 3 — Clone or update the repo (idempotent; safe to re-run)
import os
BRANCH = 'claude/setup-gpu-video-testing-JhgUH'
REPO   = 'https://github.com/muwafagq/playbook-program.git'
DEST   = '/content/playbook'

if os.path.isdir(DEST + '/.git'):
    print('Repo exists — fetching latest...')
    !git -C {DEST} fetch -q origin {BRANCH}
    !git -C {DEST} checkout -q {BRANCH}
    !git -C {DEST} reset -q --hard origin/{BRANCH}
else:
    !rm -rf {DEST}
    !git clone -q --branch {BRANCH} {REPO} {DEST}

import sys
os.chdir(DEST); sys.path.insert(0, DEST)
head = !git -C {DEST} log --oneline -1
print('Branch:', BRANCH, '| HEAD:', head[0])

In [ ]:
# Cell 4 — Configure .env (copy baseline + inject your API key). NO file patching.
import shutil, os
shutil.copy('/content/playbook/baseline.env', '/content/playbook/.env')

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    ROBOFLOW_API_KEY = input('Paste ROBOFLOW_API_KEY: ').strip()

with open('/content/playbook/.env', 'a') as f:
    f.write(f'\nROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')

# Show the homography-relevant settings actually in effect (sanity check)
import re
env = open('/content/playbook/.env').read()
for k in ['FIELD_MODEL_ID','PLAYER_MODEL_ID','KP_CONF','RANSAC_REPROJ_THRESH','H_OPTFLOW_BRIDGE']:
    m = re.search(rf'^{k}=(.*)$', env, re.M)
    print(f'{k:24s}= {m.group(1) if m else "(default)"}')
print('\n.env configured — no source files modified.')

In [ ]:
# Cell 5 — Upload your video (renamed to a space-free path to avoid path bugs)
import os
from google.colab import files as colab_files
print('Select your .mp4 ...')
up = colab_files.upload()
src = '/content/' + list(up.keys())[0]
VIDEO_INPUT = '/content/input_video.mp4'
os.replace(src, VIDEO_INPUT)
print('Video ready:', VIDEO_INPUT)

In [ ]:
# Cell 6 — Run the pipeline on GPU
import os, sys, subprocess
OUT_DIR = '/content/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

run_env = os.environ.copy()
run_env['DEVICE'] = 'cuda'
run_env['ONNXRUNTIME_EXECUTION_PROVIDERS'] = 'CUDAExecutionProvider'
try:
    from google.colab import userdata
    run_env['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    pass

cmd = [sys.executable, '/content/playbook/main.py',
       '--source-video', VIDEO_INPUT, '--out-dir', OUT_DIR, '--enable-team']

print('Launching:', VIDEO_INPUT, '\n')
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1, env=run_env)
for line in p.stdout:
    print(line, end='')
p.wait()
print('\n', 'DONE' if p.returncode == 0 else f'FAILED (code {p.returncode})')

In [ ]:
# Cell 7 — KPI summary
import json, os, pandas as pd
OUT_DIR = '/content/outputs'
kpi = os.path.join(OUT_DIR, 'kpi_summary.json')
if os.path.exists(kpi):
    print(json.dumps(json.load(open(kpi)), indent=2))
csv = os.path.join(OUT_DIR, 'per_frame_tracks.csv')
if os.path.exists(csv):
    df = pd.read_csv(csv)
    print(f'\nCSV: {len(df):,} rows | {df.frame.nunique()} frames | {df.track_id.nunique()} IDs')
    display(df.head())

In [ ]:
# Cell 8 — Download outputs
import os
from google.colab import files as colab_files
OUT_DIR = '/content/outputs'
for fn in ['annotated.mp4','per_frame_tracks.csv','kpi_summary.json','kpi_summary.csv']:
    fp = os.path.join(OUT_DIR, fn)
    if os.path.exists(fp):
        colab_files.download(fp)
    else:
        print('skip (missing):', fp)